# Filament Thresholding

**Objective:** use only the difficult COSPAR ISWAT cases to choose provisional,
interpretable thresholds for separating filament false alarms from coronal holes.

This is a visual threshold laboratory, not a population-level accuracy estimate.
It deliberately does not use Miracle data or the Kislovodsk catalogue.


## 1. Setup

The three tentative filament votes reproduce the criteria examined in the talk:

- high component elongation;
- low absolute skew of the strong-field HMI distribution;
- low mean normalized AIA 304 brightness.

HMI and 304 are expected to overlap strongly between classes. Their weights may be
set to zero. A missing HMI value, especially near the limb, is excluded from that
component's weighted denominator.


In [ ]:
import json
import os
import re
import sys
from functools import lru_cache
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/helio_n_matplotlib")

import astropy.units as u
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL.Image
import sunpy.map
from IPython.display import clear_output, display
from reproject import reproject_interp
from scipy import ndimage
from sunpy.coordinates import frames
from sunpy.map.maputils import (
    all_coordinates_from_map,
    coordinate_is_on_solar_disk,
)
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Library").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "Library").exists(), PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Library.IO import prepare_fits
from Library.Model import load_trained_model
from Library.Processing import (
    fits_to_pmap,
    get_postprocessing_params,
    pmap_to_mask,
)


In [ ]:
ISWAT_ROOT = (
    Path.home() / "Developer" / " Misc" / "COSPAR ISWAT CH dataset"
)
CACHE_ROOT = PROJECT_ROOT / "Outputs" / "Filaments" / "ISWAT Thresholding"
RESAMPLED_ROOT = CACHE_ROOT / "Resampled 1024"
PMAP_ROOT = CACHE_ROOT / "Probability Maps"
LABEL_ROOT = CACHE_ROOT / "Component Labels"
HMI_ROOT = CACHE_ROOT / "HMI Radial"
FEATURES_PATH = CACHE_ROOT / "ISWAT Component Features LOS12G.parquet"
MANUAL_LABELS_PATH = CACHE_ROOT / "ISWAT Manual Component Labels.parquet"
THRESHOLDS_PATH = CACHE_ROOT / "ISWAT Thresholds.json"

ARCHITECTURE_ID = "A2"
DATE_RANGE_ID = "D1"
POSTPROCESSING_ID = "P1"

TARGET_SIZE = 1024
HMI_SMOOTH_SIGMA_PX = 1.0
HMI_STRONG_FIELD_G = 12.0
HMI_MAX_THETA_DEG = 60.0
REBUILD_FEATURES = False

for directory in [
    CACHE_ROOT,
    RESAMPLED_ROOT,
    PMAP_ROOT,
    LABEL_ROOT,
    HMI_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

assert ISWAT_ROOT.exists(), ISWAT_ROOT
assert (
    PROJECT_ROOT
    / "Outputs"
    / "Models"
    / f"{ARCHITECTURE_ID}{DATE_RANGE_ID}.keras"
).exists()

{
    "ISWAT_ROOT": ISWAT_ROOT,
    "model": f"{ARCHITECTURE_ID}{DATE_RANGE_ID}",
    "postprocessing": POSTPROCESSING_ID,
    "cache": CACHE_ROOT,
}


## 2. Index the ISWAT cases

Filenames are matched by observation minute. The annotated PNG is used only for
side-by-side expert review; it is not treated as a pixel mask.


In [ ]:
def parse_aia_date_key(path):
    match = re.search(
        r"(\d{4})-(\d{2})-(\d{2})T(\d{2})_(\d{2})",
        path.name,
    )
    assert match is not None, path
    return "".join(match.groups()[:3]) + "_" + "".join(match.groups()[3:])


def parse_hmi_date_key(path):
    match = re.search(
        r"(\d{4})\.(\d{2})\.(\d{2})_(\d{2})_(\d{2})",
        path.name,
    )
    assert match is not None, path
    return "".join(match.groups()[:3]) + "_" + "".join(match.groups()[3:])


In [ ]:
aia193_by_date = {
    parse_aia_date_key(path): path
    for path in sorted((ISWAT_ROOT / "193").glob("*.fits"))
}
aia304_by_date = {
    parse_aia_date_key(path): path
    for path in sorted((ISWAT_ROOT / "304").glob("*.fits"))
}
hmi_by_date = {
    parse_hmi_date_key(path): path
    for path in sorted((ISWAT_ROOT / "HMI").glob("*.fits"))
}
annotation_by_date = {
    parse_aia_date_key(path): path
    for path in sorted(
        (ISWAT_ROOT / "Coronal Hole Labels" / "Labels").glob("*-annot.png")
    )
}

common_dates = sorted(
    set(aia193_by_date)
    & set(aia304_by_date)
    & set(annotation_by_date)
)
assert common_dates

hmi_matches = {}
hmi_offsets_minutes = {}
for date_key in common_dates:
    target_time = pd.to_datetime(date_key, format="%Y%m%d_%H%M")
    hmi_key = min(
        hmi_by_date,
        key=lambda candidate: abs(
            pd.to_datetime(candidate, format="%Y%m%d_%H%M")
            - target_time
        ),
    )
    offset = abs(
        pd.to_datetime(hmi_key, format="%Y%m%d_%H%M")
        - target_time
    )
    assert offset <= pd.Timedelta(minutes=15), (date_key, hmi_key)
    hmi_matches[date_key] = hmi_by_date[hmi_key]
    hmi_offsets_minutes[date_key] = offset.total_seconds() / 60

iswat_cases = pd.DataFrame(
    {
        "aia193_source": [str(aia193_by_date[key]) for key in common_dates],
        "aia304_source": [str(aia304_by_date[key]) for key in common_dates],
        "hmi_source": [str(hmi_matches[key]) for key in common_dates],
        "hmi_offset_minutes": [
            hmi_offsets_minutes[key] for key in common_dates
        ],
        "annotation_path": [
            str(annotation_by_date[key]) for key in common_dates
        ],
    },
    index=pd.Index(common_dates, name="date_key"),
)

print(f"Indexed {len(iswat_cases)} complete ISWAT cases.")
iswat_cases


## 3. Prepare 1024-pixel FITS

Resampling is cached. Delete the cache directory only when the source data or
resampling method changes.


In [ ]:
def resample_to_cache(source_path, channel, date_key):
    destination = RESAMPLED_ROOT / f"{channel}_{date_key}.fits"
    if not destination.exists():
        source_map = sunpy.map.Map(source_path)
        if source_map.data.shape != (TARGET_SIZE, TARGET_SIZE):
            source_map = source_map.resample(
                u.Quantity([TARGET_SIZE, TARGET_SIZE], u.pixel)
            )
        source_map.save(destination, overwrite=True)
    return str(destination)


In [ ]:
for date_key, row in tqdm(
    iswat_cases.iterrows(),
    total=len(iswat_cases),
    desc="Resampling ISWAT",
):
    iswat_cases.loc[date_key, "fits_path"] = resample_to_cache(
        row.aia193_source,
        "AIA193",
        date_key,
    )
    iswat_cases.loc[date_key, "aia304_path"] = resample_to_cache(
        row.aia304_source,
        "AIA304",
        date_key,
    )
    iswat_cases.loc[date_key, "hmi_path"] = resample_to_cache(
        row.hmi_source,
        "HMI",
        date_key,
    )

iswat_cases[
    ["fits_path", "aia304_path", "hmi_path", "annotation_path"]
]


## 4. Load the existing segmentation model

ISWAT supplies the images and expert descriptions. Candidate regions still come
from the repository's existing 193 Å segmentation model.


In [ ]:
segmentation_model = load_trained_model(
    ARCHITECTURE_ID,
    DATE_RANGE_ID,
)
postprocessing = get_postprocessing_params(POSTPROCESSING_ID)
segmentation_model


## 5. Define the component measurements

Elongation is `1 - minor_axis / major_axis`. The HMI metric is the absolute
standardized third moment after smoothing, the 60° cutoff, radial correction,
and radial correction. The 12 G strong-field cutoff is applied to the smoothed
line-of-sight field before radial correction. AIA 304 brightness is the component
mean after the repository's ordinary per-image normalization.


In [ ]:
def component_elongation(component):
    rows, columns = np.where(component)
    if len(rows) < 2:
        return 0.0

    rows = rows - rows.mean()
    columns = columns - columns.mean()
    covariance = np.array(
        [
            [np.mean(columns * columns), np.mean(columns * rows)],
            [np.mean(columns * rows), np.mean(rows * rows)],
        ]
    )
    eigenvalues = np.linalg.eigvalsh(covariance)
    major = np.sqrt(max(eigenvalues[-1], 0.0))
    minor = np.sqrt(max(eigenvalues[0], 0.0))
    return float(1.0 - minor / major) if major > 0 else 0.0


In [ ]:
def absolute_skew(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    standard_deviation = values.std()
    if standard_deviation == 0:
        return np.nan
    centered = values - values.mean()
    return float(abs(np.mean(centered**3) / standard_deviation**3))


In [ ]:
def prepare_radial_hmi(aia_map, hmi_path):
    hmi_map = sunpy.map.Map(hmi_path)
    reprojected, _ = reproject_interp(
        hmi_map,
        aia_map.wcs,
        shape_out=aia_map.data.shape,
    )
    hmi_los = np.flipud(reprojected.astype(np.float32))
    hmi_los = np.nan_to_num(hmi_los, nan=0.0)

    coordinates = all_coordinates_from_map(aia_map)
    helioprojective = coordinates.transform_to(
        frames.Helioprojective(observer=aia_map.observer_coordinate)
    )
    rho = np.sqrt(
        helioprojective.Tx.to_value(u.deg) ** 2
        + helioprojective.Ty.to_value(u.deg) ** 2
    )
    solar_radius = aia_map.rsun_obs.to_value(u.deg)
    theta = np.arcsin(np.clip(rho / solar_radius, 0.0, 1.0))
    theta_degrees = np.flipud(np.rad2deg(theta))
    mu = np.flipud(np.cos(theta).astype(np.float32))
    disk = np.flipud(coordinate_is_on_solar_disk(coordinates))
    valid = disk & (theta_degrees <= HMI_MAX_THETA_DEG) & (mu > 0)

    valid_float = valid.astype(float)
    smoothed_numerator = ndimage.gaussian_filter(
        hmi_los * valid_float,
        sigma=HMI_SMOOTH_SIGMA_PX,
    )
    smoothed_denominator = ndimage.gaussian_filter(
        valid_float,
        sigma=HMI_SMOOTH_SIGMA_PX,
    )
    smoothed_los = np.zeros_like(hmi_los, dtype=np.float32)
    supported = smoothed_denominator > 0
    smoothed_los[supported] = (
        smoothed_numerator[supported] / smoothed_denominator[supported]
    )

    radial = np.zeros_like(hmi_los, dtype=np.float32)
    radial[valid] = smoothed_los[valid] / mu[valid]
    return radial, smoothed_los, valid


## 6. Extract and cache all ISWAT component features

This is the only expensive analysis cell. It caches probability maps, connected
component labels, radial HMI maps, and the resulting feature table.


In [ ]:
def extract_case_features(date_key, row):
    aia_map, aia193 = prepare_fits(row.fits_path)
    _, aia304 = prepare_fits(row.aia304_path)

    probability_path = PMAP_ROOT / f"{date_key}.npy"
    if probability_path.exists():
        probability_map = np.load(probability_path)
    else:
        probability_map = fits_to_pmap(segmentation_model, aia193)
        np.save(probability_path, probability_map)

    candidate_mask = pmap_to_mask(
        probability_map,
        smoothing_params=postprocessing,
    ).astype(bool)
    labels, component_count = ndimage.label(
        candidate_mask,
        structure=np.ones((3, 3), dtype=int),
    )
    label_path = LABEL_ROOT / f"{date_key}.npy"
    np.save(label_path, labels.astype(np.int32))

    radial_hmi, smoothed_hmi_los, hmi_valid = prepare_radial_hmi(
        aia_map,
        row.hmi_path,
    )
    hmi_path = HMI_ROOT / f"{date_key}.npy"
    np.save(hmi_path, radial_hmi)

    records = []
    for component_id in range(1, component_count + 1):
        component = labels == component_id
        strong_hmi = radial_hmi[
            component
            & hmi_valid
            & (np.abs(smoothed_hmi_los) >= HMI_STRONG_FIELD_G)
        ]
        centroid_row, centroid_column = ndimage.center_of_mass(component)
        records.append(
            {
                "date_key": date_key,
                "component_id": component_id,
                "area_px": int(component.sum()),
                "elongation": component_elongation(component),
                "hmi_abs_skew": absolute_skew(strong_hmi),
                "hmi_strong_pixels": int(strong_hmi.size),
                "aia304_mean": float(aia304[component].mean()),
                "centroid_row": float(centroid_row),
                "centroid_column": float(centroid_column),
                "labels_path": str(label_path),
                "hmi_radial_path": str(hmi_path),
            }
        )
    return records


In [ ]:
if REBUILD_FEATURES or not FEATURES_PATH.exists():
    feature_records = []
    for date_key, row in tqdm(
        iswat_cases.iterrows(),
        total=len(iswat_cases),
        desc="Extracting ISWAT features",
    ):
        feature_records.extend(extract_case_features(date_key, row))
    component_features = pd.DataFrame(feature_records)
    component_features.to_parquet(FEATURES_PATH, index=False)
else:
    component_features = pd.read_parquet(FEATURES_PATH)
    missing_dates = sorted(
        set(iswat_cases.index)
        - set(component_features["date_key"].unique())
    )
    if missing_dates:
        feature_records = []
        for date_key in tqdm(
            missing_dates,
            desc="Extracting missing ISWAT features",
        ):
            feature_records.extend(
                extract_case_features(
                    date_key,
                    iswat_cases.loc[date_key],
                )
            )
        component_features = pd.concat(
            [
                component_features,
                pd.DataFrame(feature_records),
            ],
            ignore_index=True,
        )
        component_features.to_parquet(FEATURES_PATH, index=False)

assert not component_features.empty
print(
    f"{len(component_features)} components across "
    f"{component_features['date_key'].nunique()} ISWAT cases."
)
component_features


## 7. Define the weighted threshold rule

Each available metric casts a binary filament vote. Weights are normalized over
the metrics available for that component. This prevents missing polar HMI data
from acting as a hidden CH vote.


In [ ]:
def apply_threshold_rule(
    features,
    elongation_threshold,
    hmi_skew_threshold,
    brightness_304_threshold,
    elongation_weight,
    hmi_weight,
    brightness_304_weight,
    decision_threshold,
):
    classified = features.copy()
    classified["elongation_vote"] = (
        classified["elongation"] >= elongation_threshold
    )
    classified["hmi_vote"] = (
        classified["hmi_abs_skew"] <= hmi_skew_threshold
    )
    classified["brightness_304_vote"] = (
        classified["aia304_mean"] <= brightness_304_threshold
    )

    metric_specs = [
        ("elongation", "elongation_vote", elongation_weight),
        ("hmi_abs_skew", "hmi_vote", hmi_weight),
        ("aia304_mean", "brightness_304_vote", brightness_304_weight),
    ]
    numerator = np.zeros(len(classified), dtype=float)
    denominator = np.zeros(len(classified), dtype=float)
    for value_column, vote_column, weight in metric_specs:
        available = classified[value_column].notna().to_numpy()
        numerator += (
            classified[vote_column].to_numpy(dtype=float)
            * available
            * weight
        )
        denominator += available * weight

    classified["filament_score"] = np.divide(
        numerator,
        denominator,
        out=np.full(len(classified), np.nan),
        where=denominator > 0,
    )
    classified["predicted_filament"] = (
        classified["filament_score"] >= decision_threshold
    )
    return classified


## 8. Interactive ISWAT threshold viewer

Magenta contours are currently classified as filaments; cyan contours are
currently retained as coronal holes. Component numbers match the table beneath
the plots.


In [ ]:
@lru_cache(maxsize=8)
def load_display_products(date_key):
    row = iswat_cases.loc[date_key]
    _, aia193 = prepare_fits(row.fits_path)
    _, aia304 = prepare_fits(row.aia304_path)
    labels = np.load(LABEL_ROOT / f"{date_key}.npy")
    radial_hmi = np.load(HMI_ROOT / f"{date_key}.npy")
    annotation = np.asarray(
        PIL.Image.open(row.annotation_path).convert("RGB")
    )
    return aia193, aia304, radial_hmi, labels, annotation


In [ ]:
def render_threshold_case(
    date_key,
    elongation_threshold,
    hmi_skew_threshold,
    brightness_304_threshold,
    elongation_weight,
    hmi_weight,
    brightness_304_weight,
    decision_threshold,
):
    case_features = component_features[
        component_features["date_key"] == date_key
    ].copy()
    classified = apply_threshold_rule(
        case_features,
        elongation_threshold,
        hmi_skew_threshold,
        brightness_304_threshold,
        elongation_weight,
        hmi_weight,
        brightness_304_weight,
        decision_threshold,
    )
    aia193, aia304, radial_hmi, labels, annotation = load_display_products(
        date_key
    )

    figure, axes = plt.subplots(2, 2, figsize=(15, 13))
    axes[0, 0].imshow(annotation)
    axes[0, 0].set_title(f"ISWAT expert annotation — {date_key}")
    axes[0, 0].axis("off")

    panels = [
        (axes[0, 1], aia193, "sdoaia193", None, None, "AIA 193"),
        (axes[1, 0], radial_hmi, "RdBu_r", -50, 50, "HMI radial field"),
        (axes[1, 1], aia304, "sdoaia304", None, None, "AIA 304"),
    ]
    for axis, image, color_map, lower, upper, title in panels:
        axis.imshow(image, cmap=color_map, vmin=lower, vmax=upper)
        for component in classified.itertuples(index=False):
            component_mask = labels == component.component_id
            color = "magenta" if component.predicted_filament else "cyan"
            axis.contour(
                component_mask,
                levels=[0.5],
                colors=[color],
                linewidths=1.4,
            )
            axis.text(
                component.centroid_column,
                component.centroid_row,
                str(component.component_id),
                color="white",
                fontsize=9,
                ha="center",
                va="center",
                bbox={"facecolor": color, "alpha": 0.7, "pad": 1},
            )
        axis.set_title(title)
        axis.axis("off")

    figure.suptitle(
        "Magenta = filament vote; cyan = retain as CH",
        fontsize=14,
    )
    figure.tight_layout()
    plt.show()

    columns = [
        "component_id",
        "area_px",
        "elongation",
        "hmi_abs_skew",
        "hmi_strong_pixels",
        "aia304_mean",
        "elongation_vote",
        "hmi_vote",
        "brightness_304_vote",
        "filament_score",
        "predicted_filament",
    ]
    display(
        classified[columns]
        .round(
            {
                "elongation": 3,
                "hmi_abs_skew": 3,
                "aia304_mean": 3,
                "filament_score": 3,
            }
        )
        .style.hide(axis="index")
    )


In [ ]:
case_selector = widgets.Dropdown(
    options=list(iswat_cases.index),
    description="ISWAT case",
    layout=widgets.Layout(width="360px"),
)

elongation_threshold_slider = widgets.FloatSlider(
    value=0.70,
    min=0.0,
    max=1.0,
    step=0.01,
    description="Elongation ≥",
    continuous_update=False,
    readout_format=".2f",
)
hmi_skew_threshold_slider = widgets.FloatSlider(
    value=0.20,
    min=0.0,
    max=3.0,
    step=0.01,
    description="|HMI skew| ≤",
    continuous_update=False,
    readout_format=".2f",
)
brightness_304_threshold_slider = widgets.FloatSlider(
    value=0.20,
    min=0.0,
    max=1.0,
    step=0.01,
    description="304 mean ≤",
    continuous_update=False,
    readout_format=".2f",
)

elongation_weight_slider = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=3.0,
    step=0.1,
    description="Elong. weight",
    continuous_update=False,
)
hmi_weight_slider = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=3.0,
    step=0.1,
    description="HMI weight",
    continuous_update=False,
)
brightness_304_weight_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=3.0,
    step=0.1,
    description="304 weight",
    continuous_update=False,
)
decision_threshold_slider = widgets.FloatSlider(
    value=0.50,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Score ≥",
    continuous_update=False,
    readout_format=".2f",
)

threshold_controls = {
    "date_key": case_selector,
    "elongation_threshold": elongation_threshold_slider,
    "hmi_skew_threshold": hmi_skew_threshold_slider,
    "brightness_304_threshold": brightness_304_threshold_slider,
    "elongation_weight": elongation_weight_slider,
    "hmi_weight": hmi_weight_slider,
    "brightness_304_weight": brightness_304_weight_slider,
    "decision_threshold": decision_threshold_slider,
}

threshold_output = widgets.interactive_output(
    render_threshold_case,
    threshold_controls,
)
display(
    widgets.VBox(
        [
            case_selector,
            widgets.HBox(
                [
                    widgets.VBox(
                        [
                            elongation_threshold_slider,
                            hmi_skew_threshold_slider,
                            brightness_304_threshold_slider,
                        ]
                    ),
                    widgets.VBox(
                        [
                            elongation_weight_slider,
                            hmi_weight_slider,
                            brightness_304_weight_slider,
                            decision_threshold_slider,
                        ]
                    ),
                ]
            ),
            threshold_output,
        ]
    )
)


## 9. Optionally record component-level judgements

These labels persist independently of the provisional threshold values and can
seed the constrained optimization later. Use `mixed` when removing the whole
connected component would also remove a real CH.


In [ ]:
if MANUAL_LABELS_PATH.exists():
    manual_label_table = pd.read_parquet(MANUAL_LABELS_PATH)
else:
    manual_label_table = pd.DataFrame(
        columns=["date_key", "component_id", "manual_label"]
    )

manual_labels = {
    (row.date_key, int(row.component_id)): row.manual_label
    for row in manual_label_table.itertuples(index=False)
}
print(f"Loaded {len(manual_labels)} manual component labels.")


In [ ]:
label_case_selector = widgets.Dropdown(
    options=list(iswat_cases.index),
    description="ISWAT case",
    layout=widgets.Layout(width="360px"),
)
label_component_selector = widgets.Dropdown(description="Component")
manual_label_selector = widgets.ToggleButtons(
    options=["CH", "filament", "mixed", "unclear"],
    description="Label",
)
save_label_button = widgets.Button(
    description="Save component label",
    button_style="success",
)
label_status = widgets.Output()


def refresh_component_options(change=None):
    component_ids = (
        component_features.loc[
            component_features["date_key"] == label_case_selector.value,
            "component_id",
        ]
        .astype(int)
        .tolist()
    )
    label_component_selector.options = component_ids


def save_component_label(button):
    key = (
        label_case_selector.value,
        int(label_component_selector.value),
    )
    manual_labels[key] = manual_label_selector.value
    records = [
        {
            "date_key": date_key,
            "component_id": component_id,
            "manual_label": label,
        }
        for (date_key, component_id), label in sorted(manual_labels.items())
    ]
    pd.DataFrame(records).to_parquet(MANUAL_LABELS_PATH, index=False)
    with label_status:
        clear_output(wait=True)
        print(
            f"Saved {key[0]} component {key[1]} as "
            f"{manual_label_selector.value!r}."
        )


label_case_selector.observe(refresh_component_options, names="value")
save_label_button.on_click(save_component_label)
refresh_component_options()

display(
    widgets.VBox(
        [
            label_case_selector,
            label_component_selector,
            manual_label_selector,
            save_label_button,
            label_status,
        ]
    )
)


## 10. Summarize the current rule

ISWAT is deliberately enriched for difficult cases, so predicted fractions here
must not be interpreted as operational prevalence.


In [ ]:
summarize_button = widgets.Button(
    description="Summarize current rule",
    button_style="info",
)
summary_output = widgets.Output()


def summarize_current_rule(button):
    classified = apply_threshold_rule(
        component_features,
        elongation_threshold_slider.value,
        hmi_skew_threshold_slider.value,
        brightness_304_threshold_slider.value,
        elongation_weight_slider.value,
        hmi_weight_slider.value,
        brightness_304_weight_slider.value,
        decision_threshold_slider.value,
    )
    case_summary = (
        classified.groupby("date_key")
        .agg(
            components=("component_id", "size"),
            predicted_filaments=("predicted_filament", "sum"),
        )
        .sort_index()
    )
    with summary_output:
        clear_output(wait=True)
        display(case_summary)
        print(
            f"Predicted filament components: "
            f"{int(classified['predicted_filament'].sum())} / "
            f"{len(classified)}"
        )

        if manual_labels:
            manual = pd.DataFrame(
                [
                    {
                        "date_key": date_key,
                        "component_id": component_id,
                        "manual_label": label,
                    }
                    for (date_key, component_id), label in manual_labels.items()
                ]
            )
            compared = classified.merge(
                manual,
                on=["date_key", "component_id"],
                how="inner",
            )
            display(
                pd.crosstab(
                    compared["manual_label"],
                    compared["predicted_filament"],
                    rownames=["manual"],
                    colnames=["predicted filament"],
                )
            )


summarize_button.on_click(summarize_current_rule)
display(widgets.VBox([summarize_button, summary_output]))


## 11. Export provisional threshold values

The exported values are explicitly tagged as ISWAT-derived and provisional.
They should be revisited after the production masks change.


In [ ]:
export_button = widgets.Button(
    description="Export thresholds",
    button_style="warning",
)
export_status = widgets.Output()


def export_thresholds(button):
    threshold_specification = {
        "source": "COSPAR ISWAT stress cases",
        "status": "provisional",
        "segmentation_model": f"{ARCHITECTURE_ID}{DATE_RANGE_ID}",
        "postprocessing": POSTPROCESSING_ID,
        "metrics": {
            "elongation": {
                "direction": ">=",
                "threshold": elongation_threshold_slider.value,
                "weight": elongation_weight_slider.value,
            },
            "hmi_abs_skew": {
                "direction": "<=",
                "threshold": hmi_skew_threshold_slider.value,
                "weight": hmi_weight_slider.value,
                "strong_field_G": HMI_STRONG_FIELD_G,
                "strong_field_space": "smoothed_los",
                "maximum_theta_deg": HMI_MAX_THETA_DEG,
                "smoothing_sigma_px": HMI_SMOOTH_SIGMA_PX,
            },
            "aia304_mean": {
                "direction": "<=",
                "threshold": brightness_304_threshold_slider.value,
                "weight": brightness_304_weight_slider.value,
            },
        },
        "decision_threshold": decision_threshold_slider.value,
    }
    THRESHOLDS_PATH.write_text(
        json.dumps(threshold_specification, indent=2)
    )
    with export_status:
        clear_output(wait=True)
        print(f"Saved {THRESHOLDS_PATH}")


export_button.on_click(export_thresholds)
display(widgets.VBox([export_button, export_status]))


## 12. Results and next decision

Record conclusions after reviewing the ISWAT cases:

- Does any single threshold behave monotonically enough to retain?
- Which components are `mixed`, making whole-component deletion unsafe?
- Does AIA 304 deserve a non-zero weight?
- Are the chosen values stable across the pre-2018 and 2018 ISWAT cases?

**Next step:** use these provisional values to constrain an automatic 2017
optimization after a defensible Kislovodsk association metric exists. Keep 2018
outside that optimization.
